# M4 · Sistemes de Big Data — RA1  
## Notebook 1 · Complejidad y escala

**Actividad de refuerzo — no evaluable**

En este notebook vamos a comprobar experimentalmente algo que en teoría puede parecer abstracto:

> **cuando aumenta el tamaño de los datos, el algoritmo utilizado empieza a importar mucho.**

No buscamos memorizar tiempos concretos, sino observar **patrones de crecimiento**.

### Objetivos
Al terminar deberías ser capaz de:

- relacionar el tamaño de entrada `n` con el tiempo de ejecución;
- distinguir experimentalmente comportamientos cercanos a `O(1)`, `O(log n)`, `O(n)` y `O(n²)`;
- comparar búsqueda lineal y búsqueda binaria;
- entender por qué ordenar previamente puede tener un coste;
- comparar una ordenación cuadrática con una implementación eficiente;
- visualizar por qué los problemas exponenciales dejan de ser abordables muy rápido;
- conectar estas ideas con el procesamiento de grandes volúmenes de datos.

---

## 1. Preparación

Usaremos únicamente librerías estándar de Python y `matplotlib`.

La función `medir()` ejecutará varias veces una función y devolverá el tiempo mediano.  
Usamos la mediana para reducir el efecto de pequeñas variaciones del entorno de ejecución.

In [ ]:
import random
import time
import math
import bisect
import statistics
import matplotlib.pyplot as plt

random.seed(42)

def medir(func, repeticiones=5):
    tiempos = []

    for _ in range(repeticiones):
        inicio = time.perf_counter()
        func()
        fin = time.perf_counter()
        tiempos.append(fin - inicio)

    return statistics.median(tiempos)

print("Entorno preparado.")

## 2. Primera idea: acceder no es lo mismo que buscar

Si conocemos la posición de un elemento en una lista, podemos acceder directamente:

```python
datos[500]
```

Pero si solo conocemos su valor, podemos necesitar recorrer la lista hasta encontrarlo.

Vamos a comparar ambas operaciones.

In [ ]:
def acceso_directo(datos, indice):
    return datos[indice]

def busqueda_lineal(datos, objetivo):
    for valor in datos:
        if valor == objetivo:
            return True
    return False

In [ ]:
tamanos = [1_000, 10_000, 100_000, 1_000_000]

tiempos_acceso = []
tiempos_lineal = []

for n in tamanos:
    datos = list(range(n))

    t_acceso = medir(lambda: acceso_directo(datos, n - 1))
    t_lineal = medir(lambda: busqueda_lineal(datos, n - 1))

    tiempos_acceso.append(t_acceso)
    tiempos_lineal.append(t_lineal)

for n, ta, tl in zip(tamanos, tiempos_acceso, tiempos_lineal):
    print(
        f"n={n:>9,} | "
        f"acceso={ta*1e6:>9.3f} µs | "
        f"búsqueda lineal={tl*1e3:>9.3f} ms"
    )

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(tamanos, tiempos_acceso, marker="o", label="Acceso directo")
plt.plot(tamanos, tiempos_lineal, marker="o", label="Búsqueda lineal")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Número de elementos (n)")
plt.ylabel("Tiempo mediano (s)")
plt.title("Acceso directo vs búsqueda lineal")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Interpreta

1. ¿El tiempo de acceso directo aumenta mucho cuando pasamos de 1.000 a 1.000.000 elementos?
2. ¿Qué ocurre con la búsqueda lineal?
3. ¿Qué complejidad aproximada asociarías a cada operación?

**Idea clave:** `O(1)` no significa que una operación tarde literalmente cero segundos.  
Significa que el número de operaciones necesarias **no crece con `n`**.

## 3. Búsqueda lineal vs búsqueda binaria

La búsqueda binaria descarta aproximadamente la mitad de los elementos en cada paso.

Pero tiene una condición importante:

> **los datos deben estar ordenados.**

Usaremos `bisect`, una implementación eficiente de búsqueda binaria incluida en Python.

In [ ]:
def busqueda_binaria(datos_ordenados, objetivo):
    posicion = bisect.bisect_left(datos_ordenados, objetivo)

    return (
        posicion < len(datos_ordenados)
        and datos_ordenados[posicion] == objetivo
    )

In [ ]:
tamanos_busqueda = [1_000, 10_000, 100_000, 1_000_000, 5_000_000]

tiempos_lineal = []
tiempos_binaria = []

for n in tamanos_busqueda:
    datos = list(range(n))
    objetivo = n - 1

    tiempos_lineal.append(
        medir(lambda: busqueda_lineal(datos, objetivo), repeticiones=3)
    )

    tiempos_binaria.append(
        medir(lambda: busqueda_binaria(datos, objetivo), repeticiones=20)
    )

for n, tl, tb in zip(tamanos_busqueda, tiempos_lineal, tiempos_binaria):
    print(
        f"n={n:>10,} | "
        f"lineal={tl*1e3:>9.3f} ms | "
        f"binaria={tb*1e6:>9.3f} µs"
    )

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(tamanos_busqueda, tiempos_lineal, marker="o", label="Lineal")
plt.plot(tamanos_busqueda, tiempos_binaria, marker="o", label="Binaria")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Número de elementos (n)")
plt.ylabel("Tiempo mediano (s)")
plt.title("Búsqueda lineal vs búsqueda binaria")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### ¿Por qué la diferencia es tan grande?

Número aproximado de pasos:

- búsqueda lineal: hasta `n`;
- búsqueda binaria: aproximadamente `log2(n)`.

Vamos a verlo.

In [ ]:
for n in [1_000, 10_000, 100_000, 1_000_000, 1_000_000_000]:
    print(
        f"n={n:>13,} | "
        f"lineal (peor caso) ≈ {n:>13,} pasos | "
        f"binaria ≈ {math.ceil(math.log2(n)):>3} pasos"
    )

## 4. Pero ordenar también cuesta

La búsqueda binaria parece claramente superior, pero los datos deben estar ordenados.

Imagina que recibimos una lista desordenada y solo queremos realizar **una búsqueda**.

Puede que ordenar todo el dataset antes de buscar no compense.

Sin embargo, si realizaremos **miles o millones de búsquedas**, ordenar o indexar previamente puede ser una inversión útil.

Vamos a medir:

1. buscar una vez linealmente;
2. ordenar + buscar binariamente;
3. realizar muchas búsquedas sobre la misma lista ya ordenada.

In [ ]:
n = 500_000
datos_desordenados = list(range(n))
random.shuffle(datos_desordenados)

objetivo = n - 1

t_lineal_una = medir(
    lambda: busqueda_lineal(datos_desordenados, objetivo),
    repeticiones=3
)

def ordenar_y_buscar():
    datos_ordenados = sorted(datos_desordenados)
    return busqueda_binaria(datos_ordenados, objetivo)

t_ordenar_buscar = medir(ordenar_y_buscar, repeticiones=3)

print(f"Una búsqueda lineal:      {t_lineal_una:.6f} s")
print(f"Ordenar + buscar binario: {t_ordenar_buscar:.6f} s")

In [ ]:
datos_ordenados = sorted(datos_desordenados)

objetivos = random.sample(range(n), 2_000)

def muchas_lineales():
    for objetivo in objetivos:
        busqueda_lineal(datos_desordenados, objetivo)

def muchas_binarias():
    for objetivo in objetivos:
        busqueda_binaria(datos_ordenados, objetivo)

t_muchas_lineales = medir(muchas_lineales, repeticiones=1)
t_muchas_binarias = medir(muchas_binarias, repeticiones=3)

print(f"2.000 búsquedas lineales: {t_muchas_lineales:.4f} s")
print(f"2.000 búsquedas binarias: {t_muchas_binarias:.4f} s")

### Conexión con Big Data

Esta idea aparece continuamente en sistemas reales:

- ordenar;
- crear índices;
- particionar;
- preagregar;
- construir estructuras auxiliares.

Todas estas operaciones tienen un coste inicial, pero pueden acelerar enormemente las consultas posteriores.

---

## 5. Ordenación: `O(n²)` frente a algoritmos eficientes

Vamos a implementar **Bubble Sort**.

No lo usamos porque sea una buena opción real.  
Precisamente lo usamos porque permite observar claramente el problema de una complejidad cuadrática.

In [ ]:
def bubble_sort(datos):
    datos = datos.copy()
    n = len(datos)

    for i in range(n):
        intercambio = False

        for j in range(0, n - i - 1):
            if datos[j] > datos[j + 1]:
                datos[j], datos[j + 1] = datos[j + 1], datos[j]
                intercambio = True

        if not intercambio:
            break

    return datos

Python incorpora `sorted()`, una implementación altamente optimizada basada en **Timsort**.

Su comportamiento es mucho más escalable que Bubble Sort.

Para Bubble Sort utilizaremos tamaños pequeños para evitar tiempos excesivos.

In [ ]:
tamanos_sort = [250, 500, 1_000, 2_000, 4_000]

tiempos_bubble = []
tiempos_python = []

for n in tamanos_sort:
    base = [random.randint(0, n * 10) for _ in range(n)]

    tiempos_bubble.append(
        medir(lambda: bubble_sort(base), repeticiones=1)
    )

    tiempos_python.append(
        medir(lambda: sorted(base), repeticiones=5)
    )

for n, tb, tp in zip(tamanos_sort, tiempos_bubble, tiempos_python):
    print(
        f"n={n:>5,} | "
        f"Bubble={tb:>9.5f} s | "
        f"sorted()={tp*1e3:>9.5f} ms"
    )

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(tamanos_sort, tiempos_bubble, marker="o", label="Bubble Sort")
plt.plot(tamanos_sort, tiempos_python, marker="o", label="Python sorted()")
plt.yscale("log")
plt.xlabel("Número de elementos (n)")
plt.ylabel("Tiempo mediano (s)")
plt.title("Ordenación cuadrática vs ordenación eficiente")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Observa el crecimiento

Si duplicamos `n`:

- un algoritmo `O(n)` tiende aproximadamente a duplicar el trabajo;
- uno `O(n log n)` crece algo más que el doble;
- uno `O(n²)` puede necesitar aproximadamente **4 veces más trabajo**.

Vamos a comprobar cuánto crece Bubble Sort en nuestro entorno.

In [ ]:
for i in range(1, len(tamanos_sort)):
    factor_n = tamanos_sort[i] / tamanos_sort[i - 1]
    factor_t = tiempos_bubble[i] / tiempos_bubble[i - 1]

    print(
        f"{tamanos_sort[i-1]:>4,} → {tamanos_sort[i]:>4,} elementos | "
        f"n × {factor_n:.1f} | "
        f"tiempo × {factor_t:.2f}"
    )

## 6. El problema exponencial

Hay problemas en los que el número de posibilidades no crece lineal ni cuadráticamente.

Por ejemplo, para un conjunto de `n` elementos existen:

\[
2^n
\]

subconjuntos posibles.

Vamos a ver cómo crece este número.

In [ ]:
for n in [10, 20, 30, 40, 50, 60]:
    print(f"n={n:>2} → {2**n:>22,} subconjuntos")

No necesitamos enumerar valores enormes para comprender el problema.

Pero sí podemos medir tamaños pequeños.

In [ ]:
def contar_subconjuntos(n):
    contador = 0

    for _ in range(2 ** n):
        contador += 1

    return contador

tamanos_exp = [10, 12, 14, 16, 18, 20]

tiempos_exp = []

for n in tamanos_exp:
    t = medir(lambda: contar_subconjuntos(n), repeticiones=1)
    tiempos_exp.append(t)

    print(
        f"n={n:>2} | "
        f"posibilidades={2**n:>10,} | "
        f"tiempo={t:.5f} s"
    )

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(tamanos_exp, tiempos_exp, marker="o")
plt.xlabel("n")
plt.ylabel("Tiempo (s)")
plt.title("Crecimiento al recorrer 2^n posibilidades")
plt.grid(True, alpha=0.3)
plt.show()

## 7. ¿Y si no podemos probar todas las posibilidades?

Aquí aparecen las **heurísticas**.

Una heurística busca una solución suficientemente buena en un tiempo razonable, sin garantizar necesariamente que sea la solución óptima.

Ejemplo conceptual:

- solución exacta: probar todas las rutas posibles del viajante;
- solución heurística: ir siempre a la ciudad no visitada más cercana.

En problemas combinatorios grandes, una solución aproximada obtenida en segundos puede ser mucho más útil que una solución óptima que tardaría un tiempo inasumible.

---

## 8. Mini-reto: predice antes de ejecutar

Antes de ejecutar la siguiente celda, intenta responder:

Si Bubble Sort tarda aproximadamente `T` segundos con 4.000 elementos, ¿cuánto esperas que tarde con 8.000?

Si el comportamiento fuese exactamente `O(n²)`:

\[
\left(\frac{8000}{4000}\right)^2 = 4
\]

Por tanto esperaríamos aproximadamente `4T`.

In [ ]:
n = 8_000
base = [random.randint(0, n * 10) for _ in range(n)]

t_8000 = medir(lambda: bubble_sort(base), repeticiones=1)

estimacion = tiempos_bubble[-1] * 4

print(f"Tiempo observado con 8.000 elementos: {t_8000:.4f} s")
print(f"Estimación a partir de O(n²):         {estimacion:.4f} s")
print(f"Relación observado / estimado:        {t_8000 / estimacion:.3f}")

> Los tiempos reales nunca tienen por qué seguir perfectamente la fórmula de Big-O.  
> Influyen el hardware, cachés, sistema operativo, datos concretos, etc.

Big-O describe principalmente **cómo escala el trabajo cuando aumenta `n`**.

---

## 9. Mini-reto personal

Prueba uno de estos retos:

### Reto A
Cambia los tamaños de búsqueda y comprueba qué ocurre con `10_000_000` elementos.

### Reto B
Busca un elemento situado al principio, en el centro y al final de una lista.

¿La búsqueda lineal tarda siempre lo mismo?

### Reto C
Prueba Bubble Sort con tamaños intermedios:

```text
3.000
5.000
6.000
```

¿El crecimiento observado se parece a `n²`?

### Reto D
Aumenta el problema exponencial desde `n=20` poco a poco.

**No subas directamente a valores enormes.**  
Detén la ejecución si empieza a tardar demasiado.

---

## 10. Conclusiones

Completa mentalmente estas frases:

**1. Cuando `n` aumenta, la elección del algoritmo...**

**2. Una búsqueda binaria puede ser muchísimo más rápida que una lineal, pero...**

**3. Un algoritmo `O(n²)` puede funcionar perfectamente con pocos datos y convertirse en un problema cuando...**

**4. En un problema exponencial, aumentar ligeramente `n`...**

**5. En Big Data, antes de comprar más hardware también deberíamos preguntarnos...**

---

### Idea final

> **Escalar no significa únicamente disponer de máquinas más potentes. También significa utilizar algoritmos, estructuras y arquitecturas capaces de crecer con los datos.**

En el siguiente notebook veremos otra dimensión de esta misma idea:

**¿qué ocurre cuando no solo cambia el algoritmo, sino también la herramienta y el formato de almacenamiento?**

→ **Pandas vs Polars · CSV vs Parquet**